<a href="https://colab.research.google.com/github/pichate-k/advance-ML/blob/main/text_mining/instructure/Adv_ML_Text_mining_prepare_lab_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Preparation

In [1]:
!git clone https://github.com/PyThaiNLP/wisesight-sentiment.git

Cloning into 'wisesight-sentiment'...
remote: Enumerating objects: 312, done.
remote: Counting objects: 100% (165/165), done.
remote: Compressing objects: 100% (119/119), done.
remote: Total 312 (delta 94), reused 89 (delta 44), pack-reused 147 (from 1)
Receiving objects: 100% (312/312), 11.14 MiB | 15.67 MiB/s, done.
Resolving deltas: 100% (174/174), done.


> Dataset Note:
> ชุดข้อมูลนี้เป็นข้อความภาษาไทยจากสื่อสังคมออนไลน์
> ที่นำมาใช้แทน customer feedback เพื่อการเรียนรู้เรื่อง sentiment classification
> ไม่ใช่ข้อมูลรีวิวสินค้าจาก marketplace โดยตรง

In [3]:
# import and Setting
import re
import pandas as pd
from pathlib import Path

RANDOM_STATE = 42
SAMPLES_PER_CLASS = 400

SOURCE_DIR = Path("wisesight-sentiment")
OUTPUT_FILE = "customer_feedback_sentiment.csv"

In [4]:
#read data
def read_sentiment_file(file_path, label):
    with open(file_path, "r", encoding="utf-8") as file:
        rows = [line.strip() for line in file if line.strip()]

    return pd.DataFrame({
        "review_text": rows,
        "sentiment": label,
        "source": "wisesight_sentiment"
    })

positive_df = read_sentiment_file(
    SOURCE_DIR / "pos.txt",
    "positive"
)

negative_df = read_sentiment_file(
    SOURCE_DIR / "neg.txt",
    "negative"
)

print("Positive:", len(positive_df))
print("Negative:", len(negative_df))

Positive: 4778
Negative: 6823


In [5]:
#Pre cleaning
def prepare_for_lab(df):
    data = df.copy()

    data["review_text"] = (
        data["review_text"]
        .astype(str)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    data = data.dropna(subset=["review_text", "sentiment"])
    data = data.drop_duplicates(subset=["review_text"])

    data["char_count"] = data["review_text"].str.len()

    data = data[data["char_count"] >= 10].copy()

    data = data.drop(columns=["char_count"])

    return data.reset_index(drop=True)

positive_clean = prepare_for_lab(positive_df)
negative_clean = prepare_for_lab(negative_df)

print("Positive after pre-cleaning:", len(positive_clean))
print("Negative after pre-cleaning:", len(negative_clean))

Positive after pre-cleaning: 4057
Negative after pre-cleaning: 6683


In [6]:
#Randomized Select data and save to CSV
positive_sample = positive_clean.sample(
    n=SAMPLES_PER_CLASS,
    random_state=RANDOM_STATE
)

negative_sample = negative_clean.sample(
    n=SAMPLES_PER_CLASS,
    random_state=RANDOM_STATE
)

lab_df = pd.concat(
    [positive_sample, negative_sample],
    ignore_index=True
)

lab_df = lab_df.sample(
    frac=1,
    random_state=RANDOM_STATE
).reset_index(drop=True)

lab_df.insert(
    0,
    "review_id",
    range(1, len(lab_df) + 1)
)

lab_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(f"Saved: {OUTPUT_FILE}")
print("Final shape:", lab_df.shape)

display(lab_df.head(10))
display(lab_df["sentiment"].value_counts())

Saved: customer_feedback_sentiment.csv
Final shape: (800, 4)


,review_id,review_text,sentiment,source
0,1,ข้างซองบุหรี่เขียนยาดองศพตัวเท่าควาย เสือกขายไ...,negative,wisesight_sentiment
1,2,ไช้ครับป๋าเขาเขาควันเยอะบอกอัตราย,negative,wisesight_sentiment
2,3,โอเคค่ะ ขอบคุณนะคะ,positive,wisesight_sentiment
3,4,อย่าโชว์เหนือ เสือไม่ชอบ,negative,wisesight_sentiment
4,5,ลดน้ำหนักอยู่,positive,wisesight_sentiment
5,6,ขี้แตกด้วยมึง กินลีโอเปนงี้ตล่อดด 555,negative,wisesight_sentiment
6,7,ผิวบอบบางยิ่งต้องดูแลเป็นพิเศษ… ใหม่! กันแดดสู...,positive,wisesight_sentiment
7,8,รถก็มีข้อผิดพลาดด้วยกันทุกค่าย... แต่ดูแลลูกค้...,negative,wisesight_sentiment
8,9,๕๕๕๕๕ เราแค่ไปกินบาบีก้อนก็คีบอุด้งหล่น น้ำกระ...,negative,wisesight_sentiment
9,10,สำหรับงานวันนี้ 5555555555555555 ดื่มหรืออาบคร...,negative,wisesight_sentiment


,count
sentiment,
negative,400
positive,400


In [7]:
#Quality Check
print("Missing values:")
display(lab_df.isnull().sum())

print("Duplicate review_text:")
print(lab_df["review_text"].duplicated().sum())

print("\nClass distribution:")
display(lab_df["sentiment"].value_counts())

print("\nRandom positive samples:")
display(
    lab_df[lab_df["sentiment"] == "positive"]
    .sample(10, random_state=42)
)

print("\nRandom negative samples:")
display(
    lab_df[lab_df["sentiment"] == "negative"]
    .sample(10, random_state=42)
)

Missing values:


,0
review_id,0
review_text,0
sentiment,0
source,0


Duplicate review_text:
0

Class distribution:


,count
sentiment,
negative,400
positive,400



Random positive samples:


,review_id,review_text,sentiment,source
404,405,ขอไอดีไลน์เซลล์ด้วยค่ะ ดอกเบี้ยถูกมากค่ะ เค้าค...,positive,wisesight_sentiment
551,552,เครื่อง 600 cc เจอเครื่อง 1200 มาสด้า2 ประหยัด...,positive,wisesight_sentiment
66,67,เลิกม้วนได้เพราะไฟฟ้าเลย,positive,wisesight_sentiment
405,406,ชอบรถ C-HR สีแดงเพราะ สวย เฉีบบ น่าสัมผัส สี่ท...,positive,wisesight_sentiment
182,183,ยาริสสวยงั้บ 🤘,positive,wisesight_sentiment
165,166,อยากกินเซ็ทนี้คะ,positive,wisesight_sentiment
668,669,ไปๆสนุกกัน,positive,wisesight_sentiment
185,186,เลี้ยงหรอ ขอบคุณค่ะ,positive,wisesight_sentiment
513,514,ชุดไหนดี เพื่อนบ้านในอณาคตสนใจม้ายยยย,positive,wisesight_sentiment
246,247,อยากกินนนน สอบเสร็จจัดไปคะ5555,positive,wisesight_sentiment



Random negative samples:


,review_id,review_text,sentiment,source
429,430,นี่นิสสันยังดูไม่ใหญ่อีกเหรอครับ..ส่วนตัวคิดว่...,negative,wisesight_sentiment
570,571,ลวกสุกี้ กับบะหมี่เป็ดย่างนานขนาดนั้นเลย หรือก...,negative,wisesight_sentiment
67,68,กุเกลียดยูเบียร์ รสชาติเหี้ย,negative,wisesight_sentiment
430,431,สงกรานต์แดกทุกปี ซื้อเรด ซื้อแบล๊คไม่ไหว กระเป...,negative,wisesight_sentiment
188,189,ไม่ได้สังเกตุอะ แต่คงไม่พาไปกินละ เข็ดละ ถึงเค...,negative,wisesight_sentiment
177,178,ถ้าลีโอร้อยกว่าบาทก็แดกน้ำองุ่นทิปโก้เอาแหละ บ้า,negative,wisesight_sentiment
654,655,สั่งแมคฟิส ได้ซอสมา1 ซอง ควรใช้วิจารณญาณในการห...,negative,wisesight_sentiment
189,190,ยังไม่หิว 7 ฮ่าๆๆๆ,negative,wisesight_sentiment
543,544,ยูเบียร์สดแย่กว่าที่คิด,negative,wisesight_sentiment
264,265,ชั้นจะเปนทาสเทอตะหลอดไปอินนิสฟรี ชั้นจะเลิกใช้...,negative,wisesight_sentiment


In [8]:
#Download file
from google.colab import files

files.download(OUTPUT_FILE)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>